# 3. Submission - XGBoost con Feature Engineering

Cargar el modelo XGBoost con feature engineering, reentrenar sobre todo el train, predecir sobre test y enviar a Kaggle.

**Modelo:** XGBoost con hiperparámetros optimizados + 11 features generadas (24 total)

**Competencia:** `playground-series-s6e2`

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
import joblib

## 2. Carga de datos

In [2]:
train = pd.read_csv("../../data/train.csv")
test = pd.read_csv("../../data/test.csv")

# Target binario
train["target"] = (train["Heart Disease"] == "Presence").astype(int)

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")

Train: (630000, 16)
Test:  (270000, 14)


## 3. Feature Engineering

Misma función `build_features()` que en el notebook de modelo.
Se aplica tanto a train como a test para garantizar consistencia.

In [3]:
def build_features(df):
    """Aplica feature engineering sobre un DataFrame con las features originales.
    Retorna el DataFrame con las features nuevas agregadas (no modifica el original).
    """
    X = df.copy()

    # ── Grupo A: Indicadores clínicos derivados ──────────────────────────────
    X["HR_reserve"] = X["Max HR"] - X["Age"]
    X["Double_Product"] = X["BP"] * X["Max HR"]
    X["Chol_HR_ratio"] = X["Cholesterol"] / (X["Max HR"] + 1)

    # ── Grupo B: Interacciones ───────────────────────────────────────────────
    X["Age_x_MaxHR"] = X["Age"] * X["Max HR"]
    X["BP_x_Chol"] = X["BP"] * X["Cholesterol"]
    X["Age_x_STdep"] = X["Age"] * X["ST depression"]

    # ── Grupo C: Transformaciones no lineales ────────────────────────────────
    X["ST_dep_sq"] = X["ST depression"] ** 2
    X["ST_dep_present"] = (X["ST depression"] > 0).astype(int)

    # ── Grupo D: Binning clínico ─────────────────────────────────────────────
    X["Age_group"] = pd.cut(
        X["Age"],
        bins=[0, 44, 54, 64, 120],
        labels=[0, 1, 2, 3]
    ).astype(int)

    X["BP_category"] = pd.cut(
        X["BP"],
        bins=[0, 119, 129, 139, 300],
        labels=[0, 1, 2, 3]
    ).astype(int)

    X["Chol_category"] = pd.cut(
        X["Cholesterol"],
        bins=[0, 199, 239, 1000],
        labels=[0, 1, 2]
    ).astype(int)

    return X


# Aplicar FE a train y test
train_fe = build_features(train)
test_fe = build_features(test)

print(f"Train FE: {train_fe.shape}")
print(f"Test FE:  {test_fe.shape}")

Train FE: (630000, 27)
Test FE:  (270000, 25)


## 4. Cargar modelo y reentrenar sobre todo el train

In [4]:
saved = joblib.load("../../models/3_xgboost_fe_best.pkl")
best_params = saved["best_params"]
model_features = saved["model_features"]
cv_score = saved["cv_score"]
cv_std = saved["cv_std"]

print(f"Hiperparámetros: {best_params}")
print(f"Features ({len(model_features)}): {model_features}")
print(f"ROC-AUC (CV): {cv_score:.4f} +/- {cv_std:.4f}")

# Reentrenar sobre todo el dataset de train
X_train = train_fe[model_features]
y_train = train_fe["target"]

model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,
    **best_params
)
model.fit(X_train, y_train, verbose=False)
print(f"\nModelo entrenado sobre {len(X_train)} muestras.")

Hiperparámetros: {'colsample_bytree': np.float64(0.749816047538945), 'gamma': np.float64(0.4753571532049581), 'learning_rate': np.float64(0.22227824312530747), 'max_depth': 7, 'min_child_weight': 5, 'n_estimators': 714, 'subsample': np.float64(0.7783331011414365)}
Features (24): ['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression', 'Sex', 'Chest pain type', 'FBS over 120', 'EKG results', 'Exercise angina', 'Slope of ST', 'Number of vessels fluro', 'Thallium', 'HR_reserve', 'Double_Product', 'Chol_HR_ratio', 'Age_x_MaxHR', 'BP_x_Chol', 'Age_x_STdep', 'ST_dep_sq', 'ST_dep_present', 'Age_group', 'BP_category', 'Chol_category']
ROC-AUC (CV): 0.9509 +/- 0.0005

Modelo entrenado sobre 630000 muestras.


## 5. Predicción sobre test

In [5]:
X_test = test_fe[model_features]
test_probs = model.predict_proba(X_test)[:, 1]

# Submission con probabilidades (formato requerido por Kaggle)
submission = pd.DataFrame({
    "id": test["id"],
    "Heart Disease": np.round(test_probs, 4)
})

print(f"Shape: {submission.shape}")
print(f"\nEstadísticas de probabilidades:")
print(submission["Heart Disease"].describe().round(4))
print(f"\nPrimeras filas:")
submission.head(10)

Shape: (270000, 2)

Estadísticas de probabilidades:
count    270000.0000
mean          0.4498
std           0.4134
min           0.0000
25%           0.0322
50%           0.3080
75%           0.9363
max           1.0000
Name: Heart Disease, dtype: float64

Primeras filas:


,id,Heart Disease
0,630000,0.9661
1,630001,0.0101
2,630002,0.9918
3,630003,0.0040
4,630004,0.0489
5,630005,0.9887
6,630006,0.0013
7,630007,0.6595
8,630008,0.9833
9,630009,0.0188


## 6. Guardar CSV

In [6]:
SUBMISSION_FILE = "3_XGB_FE_submission.csv"
submission.to_csv(SUBMISSION_FILE, index=False)

# Verificar
check = pd.read_csv(SUBMISSION_FILE)
print(f"Archivo: {SUBMISSION_FILE}")
print(f"Shape: {check.shape}")
print(f"Columnas: {list(check.columns)}")
print(f"IDs: {check['id'].min()} - {check['id'].max()}")
check.head()

Archivo: 3_XGB_FE_submission.csv
Shape: (270000, 2)
Columnas: ['id', 'Heart Disease']
IDs: 630000 - 899999


,id,Heart Disease
0,630000,0.9661
1,630001,0.0101
2,630002,0.9918
3,630003,0.0040
4,630004,0.0489


## 7. Submit a Kaggle

In [7]:
COMPETITION = "playground-series-s6e2"
MESSAGE = "XGBoost FE - 24 features (13 originales + 11 generadas) - RandomizedSearchCV"

!kaggle competitions submit -c {COMPETITION} -f {SUBMISSION_FILE} -m "{MESSAGE}"

Successfully submitted to Predicting Heart Disease



  0%|          | 0.00/3.83M [00:00<?, ?B/s]
  0%|          | 16.0k/3.83M [00:00<00:53, 74.8kB/s]
  5%|▍         | 192k/3.83M [00:00<00:05, 661kB/s]  
 14%|█▍        | 544k/3.83M [00:00<00:02, 1.16MB/s]
 49%|████▉     | 1.88M/3.83M [00:00<00:00, 4.38MB/s]
100%|██████████| 3.83M/3.83M [00:02<00:00, 1.90MB/s]
